# Chapter 10: the brewpub P&L with every line open to change

A synthetic 10 hL brewpub in synthetic State S. Change any input in `PLAN` and rerun.
Read the break-even as a steady state figure, never as year one.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
import math

In [ ]:
PLAN = dict(cast_hl=750, loss=0.10, glass_l=0.33, glass_price=330, vat=0.10, excise_hl=3000,
            food_per_hl=50000, food_tax=0.05, food_cost=0.35,
            malt_kg_hl=20.45, malt_price=110, hops=600, yeast=200, co2_cleaning=300, power=400, water=100,
            rent=1_80_00_000, staff=1_80_00_000, licence=15_00_000, other=1_50_00_000,
            kit=3_00_00_000, kit_life=10, fitout=5_00_00_000, fitout_life=7)

def pub(**change):
    p = {**PLAN, **change}
    sold = p['cast_hl'] * (1 - p['loss'])
    beer_net = 100 / p['glass_l'] * p['glass_price'] / (1 + p['vat']) - p['excise_hl']
    var_cast = p['malt_kg_hl'] * p['malt_price'] + p['hops'] + p['yeast'] + p['co2_cleaning'] + p['power'] + p['water']
    var_sold = var_cast * p['cast_hl'] / sold
    food_net = p['food_per_hl'] / (1 + p['food_tax'])
    food_contrib = food_net * (1 - p['food_cost'])
    # Contribution a hL is rounded to the paisa and depreciation to the rupee, as the chapter prints them;
    # profit is then sold x contribution less fixed costs.
    contrib = r2(beer_net - var_sold + food_contrib)
    dep = round(p['kit'] / p['kit_life'] + p['fitout'] / p['fitout_life'])
    fixed = p['rent'] + p['staff'] + p['licence'] + p['other'] + dep
    return dict(sold=sold, beer_net=beer_net, var_cast=var_cast, var_sold=var_sold, food_net=food_net,
                food_contrib=food_contrib, contrib=contrib, dep=dep, fixed=fixed,
                revenue=sold * (beer_net + p['excise_hl'] + food_net), profit=sold * contrib - fixed,
                breakeven=fixed / contrib)

base = pub()
print('net a hL sold', inr(base['beer_net']), '; contribution a hL sold', inr(base['contrib']))
print('revenue', inr(base['revenue'], 0), '; fixed', inr(base['fixed'], 0), '; profit', inr(base['profit'], 0),
      f"= {base['profit'] / base['revenue']:.1%} (steady state); break-even {base['breakeven']:.0f} hL")

In [ ]:
glasses_year = 675 * 100 / 0.33
p = PLAN
pnl = pd.Series({
    'Beer sales before VAT': base['sold'] * (base['beer_net'] + p['excise_hl']),
    'Food sales before tax': base['sold'] * base['food_net'],
    'Excise': base['sold'] * p['excise_hl'],
    'Beer variable cost': base['var_cast'] * p['cast_hl'],
    'Food cost': base['sold'] * base['food_net'] * p['food_cost'],
    'Contribution': base['sold'] * base['contrib'],
    'Fixed costs': base['fixed'],
    'Profit before interest and tax': base['profit']})
print(f'{glasses_year:,.0f} glasses a year, {glasses_year / 360:.0f} a day')
pnl.map(lambda x: inr(x, 0))

## Move one thing at a time (the table in 10.6)

In [ ]:
cases = {'Plan': {}, 'Menu price 10 percent lower': dict(glass_price=297), 'Menu price 10 percent higher': dict(glass_price=363),
         'Food spend 20 percent lower': dict(food_per_hl=40000), 'Rent 10 percent higher': dict(rent=1_98_00_000),
         'Malt price 20 percent higher': dict(malt_price=132), 'Cast to glass loss of 15 percent': dict(loss=0.15),
         'Packaged beer excise': dict(excise_hl=13860)}
sens = pd.DataFrame({k: pub(**v) for k, v in cases.items()}).T
sens = pd.DataFrame({'contribution a hL sold': sens.contrib.round(2), 'break-even hL': sens.breakeven.round(0),
                     'profit, lakh': (sens.profit / 1e5).round(2)})
sens

In [ ]:
beer_contrib = base['sold'] * r2(base['beer_net'] - base['var_sold'])
hurdle = 8_00_00_000 * 0.12
breakeven_with_hurdle = math.ceil((base['fixed'] + hurdle) / base['contrib'])   # whole hL needed
payback = 8_00_00_000 / (base['profit'] + base['dep'])
print('beer contributes', inr(beer_contrib, 0), f"({beer_contrib / (base['sold'] * base['contrib']):.0%} of contribution)")
print('with a 12 percent capital charge the pub needs', breakeven_with_hurdle, 'hL; payback', round(payback, 1), 'years')
print('over the bar against packaged:', round(base['beer_net'] / 12600, 1), 'times')